# Job Application Extractor Experiment
This notebook allows you to test the `EmailExtractor` with individual email data to see how the current model performs.

In [ ]:
import sys
import os
from pathlib import Path
import json

# 1. Setup paths to import from the 'app' package
current_path = Path(os.getcwd()).resolve()

# Look for 'smart-job-tracker' in the path or as a subdirectory
smart_job_tracker_dir = None
if (current_path / "smart-job-tracker").exists():
    smart_job_tracker_dir = current_path / "smart-job-tracker"
elif current_path.name == "smart-job-tracker":
    smart_job_tracker_dir = current_path
else:
    # Walk up to find it
    for p in [current_path] + list(current_path.parents):
        if (p / "smart-job-tracker").exists():
            smart_job_tracker_dir = p / "smart-job-tracker"
            break

if smart_job_tracker_dir:
    if str(smart_job_tracker_dir) not in sys.path:
        sys.path.insert(0, str(smart_job_tracker_dir))
    os.chdir(smart_job_tracker_dir)
    print(f"✅ Environment Ready. Working directory: {os.getcwd()}")
else:
    print("❌ Error: Could not find 'smart-job-tracker' directory.")

# 2. Check for missing dependencies
missing = []
deps = {
    "sqlmodel": "sqlmodel", 
    "streamlit": "streamlit", 
    "pydantic": "pydantic", 
    "bs4": "beautifulsoup4", 
    "googleapiclient": "google-api-python-client", 
    "google_auth_oauthlib": "google-auth-oauthlib", 
    "tenacity": "tenacity", 
    "dateutil": "python-dateutil", 
    "yaml": "pyyaml", 
    "dotenv": "python-dotenv"
}
for mod, pkg in deps.items():
    try:
        __import__(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"⚠️ Missing dependencies: {', '.join(missing)}")
    print("To fix this, ensure you are using the Poetry virtual environment kernel, or run:")
    print(f"!pip install {' '.join(missing)}")
else:
    print("✅ All dependencies found.")

from app.services.extractor import EmailExtractor
from app.models import ApplicationStatus

In [ ]:
# 3. Initialize the Extractor
extractor = EmailExtractor()
print("Extractor initialized successfully.")

In [ ]:
# 4. Define a function to test extraction
def test_extraction(sender, subject, body):
    print(f"\n{'='*50}")
    print(f"SENDER:  {sender}")
    print(f"SUBJECT: {subject}")
    print(f"{'='*50}")
    
    result = extractor.extract(subject, sender, body)
    
    print("\n--- EXTRACTED DATA ---")
    print(json.dumps(result.model_dump(), indent=4, default=str))
    return result

# 5. Run a few experiments
test_emails = [
    {
        "sender": "Google <noreply@google.com>",
        "subject": "Your application for Senior Software Engineer",
        "body": "Hi! Thank you for applying to Google. We have received your application for the Senior Software Engineer role and are currently reviewing it. We will be in touch soon!"
    },
    {
        "sender": "Amazon Recruiting <no-reply@amazon.com>",
        "subject": "Interview Invitation: Software Development Engineer",
        "body": "We are excited to invite you for an interview for the Software Development Engineer position. Please let us know your availability for a 60-minute technical interview next week."
    },
    {
        "sender": "Startup Inc <hiring@startup.io>",
        "subject": "Update on your application",
        "body": "Unfortunately, we have decided not to move forward with your application at this time. We wish you the best of luck in your search."
    }
]

for email in test_emails:
    test_extraction(email['sender'], email['subject'], email['body'])

In [ ]:
# 6. Real Email Processing
from app.services.gmail import get_gmail_service, batch_get_message_bodies, clean_html_for_llm

def fetch_and_test_real_emails(sender_email=None, date_query=None):
    """
    Fetches real emails matching the sender and date and runs the extractor.
    sender_email: e.g., 'recruiting@peter-park.de'
    date_query: e.g., 'after:2026/03/05 before:2026/03/07'
    """
    try:
        service = get_gmail_service()
        query = ""
        if sender_email:
            query += f"from:{sender_email} "
        if date_query:
            query += f"{date_query}"
        
        print(f"Searching Gmail with query: {query}")
        results = service.users().messages().list(userId='me', q=query, maxResults=5).execute()
        messages = results.get('messages', [])
        
        if not messages:
            print("No emails found matching the query.")
            return
        
        msg_ids = [m['id'] for m in messages]
        full_msgs = batch_get_message_bodies(service, msg_ids)
        
        for msg in full_msgs:
            print(f"\n{'='*60}")
            print(f"SENDER:  {msg['sender']}")
            print(f"SUBJECT: {msg['subject']}")
            print(f"DATE:    {msg['date']}")
            print(f"{'='*60}")
            
            cleaned_html = clean_html_for_llm(msg.get('html', ''))
            extraction = extractor.extract(msg['subject'], msg['sender'], msg['text'], cleaned_html)
            
            print("\n--- EXTRACTED DATA ---")
            print(json.dumps(extraction.model_dump(), indent=4, default=str))
            
    except Exception as e:
        print(f"Error fetching or processing emails: {e}")

print("Function 'fetch_and_test_real_emails' is ready to use.")

In [ ]:
# 7. Run Real Experiment
# Fill these in to test with real data!
TARGET_SENDER = "notifications@auto1-group.com" # e.g. "recruiting@company.com"
TARGET_DATE = "after:2023-07-11 before:2023-07-19"   # e.g. "after:2026/03/01 before:2026/03/10"

if TARGET_SENDER or TARGET_DATE:
    fetch_and_test_real_emails(TARGET_SENDER, TARGET_DATE)
else:
    print("Please provide a TARGET_SENDER or TARGET_DATE to run a real experiment.")

In [ ]:
# 8. Fetch Data from the Database for a Specific Company
from sqlmodel import Session, select
from app.core.database import engine
from app.models import JobApplication, ApplicationEventLog, Interview, Assessment, Offer

def fetch_company_data_from_db(company_name):
    """
    Fetches all applications and related data for a specific company from the database.
    """
    with Session(engine) as session:
        # 1. Fetch Application
        stmt = select(JobApplication).where(JobApplication.company_name == company_name)
        apps = session.exec(stmt).all()
        
        if not apps:
            print(f"No applications found for company: {company_name}")
            return
        
        for app in apps:
            print(f"\n{'='*60}")
            print(f"COMPANY:  {app.company_name}")
            print(f"POSITION: {app.position}")
            print(f"STATUS:   {app.status}")
            print(f"ID:       {app.id}")
            print(f"{'='*60}")
            
            # 2. Fetch History
            history = session.exec(select(ApplicationEventLog).where(ApplicationEventLog.application_id == app.id).order_by(ApplicationEventLog.event_date.desc())).all()
            print(f"\n--- HISTORY ({len(history)} events) ---")
            for h in history:
                print(f"[{h.event_date}] {h.old_status} -> {h.new_status}: {h.summary}")
            
            # 3. Fetch Interviews
            interviews = session.exec(select(Interview).where(Interview.application_id == app.id)).all()
            if interviews:
                print(f"\n--- INTERVIEWS ({len(interviews)}) ---")
                for i in interviews:
                    print(f"- {i.interview_date}: {i.interviewer or 'Unknown interviewer'} @ {i.location or 'No location'}")
            
            # 4. Fetch Assessments
            assessments = session.exec(select(Assessment).where(Assessment.application_id == app.id)).all()
            if assessments:
                print(f"\n--- ASSESSMENTS ({len(assessments)}) ---")
                for a in assessments:
                    print(f"- Due: {a.due_date}: {a.type or 'Assessment'}")
            
            # 5. Fetch Offers
            offers = session.exec(select(Offer).where(Offer.application_id == app.id)).all()
            if offers:
                print(f"\n--- OFFERS ({len(offers)}) ---")
                for o in offers:
                    print(f"- {o.offer_date}: Salary: {o.salary or 'N/A'}")

# Example usage:
TARGET_COMPANY = "AUTO1 Group"
fetch_company_data_from_db(TARGET_COMPANY)